# 01 — Test LM Studio Connectivity

Goal: Verify local LM Studio is reachable and can generate responses.
LM Studio serves an OpenAI-compatible API at `http://localhost:1234/v1`.

Prerequisites:
- LM Studio running with a model loaded
- Default port 1234 (adjust `BASE_URL` if different)

In [1]:
import os, json
import aiohttp

# Config — edit if your LM Studio runs on a different port
BASE_URL = os.getenv("LM_STUDIO_BASE_URL", "http://localhost:1234/v1")

def extract_content(choice: dict) -> str:
    """Extract response text from a chat completion choice,
    falling back to reasoning_content if content is empty."""
    msg = choice.get("message", {})
    content = msg.get("content") or ""
    if content.strip():
        return content
    reasoning = msg.get("reasoning_content") or ""
    return reasoning

print(f"Target: {BASE_URL}")

Target: http://localhost:1234/v1


## 1.1 Check server health

In [2]:
async def check_server():
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(
                f"{BASE_URL}/models",
                timeout=aiohttp.ClientTimeout(total=5.0)
            ) as resp:
                print(f"Status: {resp.status}")
                modelList= []
                if resp.status == 200:
                    models = await resp.json()
                    print("Available models:")
                    
                    for m in models.get("data", []):
                        print(f"  - {m['id']}")
                        modelList.append(m['id'])
                    return modelList
                    
                else:
                    text = await resp.text()
                    print("Response:", text[:500])
                    return modelList
    except Exception as e:
        print(f"ERROR: Cannot reach LM Studio at {BASE_URL}")
        print(f"Details: {e}")
        print("\nTroubleshooting:")
        print("  1. Is LM Studio running?")
        print("  2. Is the Local Server toggle ON?")
        print("  3. Is a model loaded?")
        print(f"  4. Is the port correct? Try: curl {BASE_URL}/models")
        return None

models= await check_server()

Status: 200
Available models:
  - gemma-4-12b-it
  - qwen/qwen3-30b-a3b-2507
  - google/gemma-4-31b-qat
  - qwen/qwen3-coder-30b
  - google/gemma-4-26b-a4b-qat
  - qwen/qwen3.5-9b
  - google/gemma-4-12b-qat
  - text-embedding-nomic-embed-text-v1.5
  - unsloth/qwen3-coder-30b-a3b-instruct
  - gemma-4-12b-coder-fable5-composer2.5-v1
  - qwen3.6-27b
  - qwen3.6-35b-a3b-mtp
  - qwen3.6-27b-mtp
  - qwen3.5-9b-mtp


In [3]:
models , len(models)

(['gemma-4-12b-it',
  'qwen/qwen3-30b-a3b-2507',
  'google/gemma-4-31b-qat',
  'qwen/qwen3-coder-30b',
  'google/gemma-4-26b-a4b-qat',
  'qwen/qwen3.5-9b',
  'google/gemma-4-12b-qat',
  'text-embedding-nomic-embed-text-v1.5',
  'unsloth/qwen3-coder-30b-a3b-instruct',
  'gemma-4-12b-coder-fable5-composer2.5-v1',
  'qwen3.6-27b',
  'qwen3.6-35b-a3b-mtp',
  'qwen3.6-27b-mtp',
  'qwen3.5-9b-mtp'],
 14)

## 1.2 Test text generation

In [4]:
async def test_generation(model: str):
    try:
        payload = {
            "model": model, #"local-model",  # LM Studio uses a placeholder model name
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Describe a Japanese anime hero in 2 sentences."}
            ],
            "temperature": 0.7,
            "max_tokens": 200,
        }
        async with aiohttp.ClientSession() as session:
            async with session.post(
                f"{BASE_URL}/chat/completions",
                json=payload,
                timeout=aiohttp.ClientTimeout(total=30.0)
            ) as resp:
                print(f"Status: {resp.status}")
                if resp.status == 200:
                    data = await resp.json()
                    content = extract_content(data["choices"][0])
                    print("\nResponse:")
                    print(content)
                    print(f"\nModel used: {data['model']}")
                    print(f"Tokens: {data['usage']}")
                else:
                    text = await resp.text()
                    print("Response:", text[:500])
    except Exception as e:
        print(f"ERROR: {e}")

await test_generation(models[-2])

Status: 200

Response:
Here's a thinking process:

1.  **Analyze User Input:**
   - **Topic:** Japanese anime hero
   - **Constraint:** Exactly 2 sentences
   - **Task:** Describe the hero

2.  **Identify Key Elements of an Anime Hero:**
   - Typical traits: Determined, emotionally driven, unique power/ability, often has a signature look (spiky hair, determined eyes), faces great odds, values friends/family, grows through adversity.
   - Need to capture the essence in a concise, evocative way.

3.  **Draft - Sentence 1 (Focus on appearance/core trait):**
   Clad in a weathered uniform and wielding a blade that hums with raw energy, he stands unyielding against impossible odds, his spiky hair catching the wind as his determined eyes never waver from the battle ahead.

4.  **Draft - Sentence 2 (Focus on motivation/theme/g

Model used: qwen3.6-27b-mtp
Tokens: {'prompt_tokens': 31, 'completion_tokens': 200, 'total_tokens': 231, 'completion_tokens_details': {'reasoning_tokens': 199}}


## 1.3 Test via langchain (OpenAI-compatible wrapper)

This is how the pipeline will actually call LM Studio.

In [5]:
from litellm import completion
from langchain_litellm import ChatLiteLLM

selModel = models[-2]  # pick from model list
print(f"Testing model: {selModel}\n")

print("=== Approach A: Direct litellm.completion() ===")
try:
    resp = completion(
        model=f"openai/{selModel}",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Describe a Japanese anime hero in 2 sentences."}
        ],
        temperature=0.7,
        max_tokens=200,
        api_base=BASE_URL,
        api_key="not-needed",
    )
    content = extract_content(resp.choices[0])
    print("Response:")
    print(content)
    print(f"\nModel: {resp.model}")
    print(f"Tokens: {resp.usage}")
except Exception as e:
    print(f"ERROR: {e}")

print("\n=== Approach B: langchain-litellm ChatLiteLLM ===")
try:
    llm = ChatLiteLLM(
        model=f"openai/{selModel}",
        api_base=BASE_URL,
        api_key="not-needed",
        temperature=0.7,
        max_tokens=200,
    )
    resp = llm.invoke([
        ("system", "You are a helpful assistant."),
        ("human", "Describe a Japanese anime hero in 2 sentences.")
    ])
    print("Response:")
    content = resp.content
    reasoning = resp.additional_kwargs.get("reasoning_content", "")
    if not content and reasoning:
        print(f"[reasoning extracted from additional_kwargs]\n{reasoning}")
    else:
        print(content)
    print(f"\nModel: {resp.response_metadata.get('model_name', 'unknown')}")
    print(f"Tokens: {resp.usage_metadata}")
except Exception as e:
    print(f"ERROR: {e}")

Testing model: qwen3.6-27b-mtp

=== Approach A: Direct litellm.completion() ===
Response:
Here's a thinking process:

1.  **Analyze User Input:**
   - **Topic:** Japanese anime hero
   - **Constraint:** Exactly 2 sentences
   - **Task:** Describe the character/figure

2.  **Identify Key Elements of an Anime Hero:**
   - Determination, unwavering resolve
   - Struggles against overwhelming odds
   - Often has a signature catchphrase or power-up moment
   - Driven by protecting others or pursuing justice/dreams
   - Visual cues: spiky hair, determined eyes, worn but iconic outfit

3.  **Draft - Sentence 1 (Focus on core traits/motivation):**
   Clad in weathered armor and fueled by an unbreakable will, he charges headfirst into impossible battles with a bright-eyed determination that never wavers.

4.  **Draft - Sentence 2 (Focus on impact/arc/theme):**
   Through every crushing

Model: qwen3.6-27b-mtp
Tokens: Usage(completion_tokens=200, prompt_tokens=31, total_tokens=231, completion_to

## Summary

| Check | Status |
|-------|--------|
| Server reachable | ⬜ |
| Text generation works | ⬜ |
| LangChain integration works | ⬜ |

If any check failed, fix the issue before proceeding to the next notebook.